# 02 - Train one MENTOR_base+L run (MKG-Y, seed 42) with the released code

**Pre-registered pass criterion (written before running).** The reference is the signed
MENTOR_base+L ("A3'") MKG-Y total test MRR, mean of 3 seeds, in `hits_bang.tsv`: **44.77**.
A single seed-42 run PASSES iff

    | test MRR at g* (g* chosen on validation) - 44.77 |  <=  0.65 points,

where 0.65 is the upper end of the observed run-to-run GPU noise (0.3-0.65 points, see the ablation table).
Otherwise FAIL. The seed-42 run of the original experiment scored 44.87 (informational only; it is not the
acceptance reference, and a seed-matched exact match is NOT expected on a different GPU).
Selection of g* uses validation ranks only.

**Inputs to attach:** (1) the release dataset (folder with `src/` and `scripts/`); (2) the existing feature
dataset that contains a folder `MKG-Y/` with `T_ctx.pt`, `V.pt`, `L_feat.pt`, `train/valid/test.txt`,
`entities.txt`, `relations.txt`, `entity2id.tsv`, `relation2id.tsv`, `image_index.tsv`.
Kaggle GPU sessions have the internet OFF; nothing here downloads anything.
Expected GPU time: about 2 minutes on a T4/P100 (the original run took 1.4 minutes).

In [16]:
import os, glob, subprocess, sys, json, time, hashlib, shutil
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")

def find_release():
    """Directory of the attached release dataset (contains scripts/verify_tables.py)."""
    hits = sorted(glob.glob(f"{INPUT}/**/scripts/verify_tables.py", recursive=True))
    assert hits, "Attach the release dataset (the folder containing scripts/verify_tables.py)."
    return os.path.dirname(os.path.dirname(hits[0]))

def run_streaming(cmd, cwd=None, env=None, tail=40):
    """Run a command, print its output line by line, return (returncode, last lines)."""
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    last = []
    for line in p.stdout:
        print(line, end="", flush=True)
        last.append(line); last = last[-tail:]
    return p.wait(), last

REL = find_release(); print("release:", REL)
NEED = ["T_ctx.pt", "V.pt", "L_feat.pt", "train.txt", "valid.txt", "test.txt", "entity2id.tsv", "relation2id.tsv", "image_index.tsv"]
cands = [dp for dp, _, fns in sorted(os.walk(INPUT)) if os.path.basename(dp) == "MKG-Y" and all(f in fns for f in NEED)]
assert cands, f"No MKG-Y folder with {NEED} under {INPUT}"
FEAT = cands[0]; print("features:", FEAT)
OUT_ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/mentor_nb"
W = f"{OUT_ROOT}/mentor"; D = f"{W}/data/MKG-Y"; RD = f"{W}/result/A3p_MKG-Y_s42"
os.makedirs(D, exist_ok=True); os.makedirs(RD, exist_ok=True)
for f in os.listdir(f"{REL}/src"):
    if f.endswith(".py"): shutil.copy(f"{REL}/src/{f}", f"{W}/{f}")
for f in os.listdir(FEAT):
    if f.endswith((".pt", ".txt", ".tsv", ".json")): shutil.copy(f"{FEAT}/{f}", f"{D}/{f}")
def sha16(*ps):
    h = hashlib.sha256()
    for p in sorted(ps): h.update(open(p, "rb").read())
    return h.hexdigest()[:16]
INPUT_HASH = sha16(*[f"{D}/{x}" for x in ("T_ctx.pt", "V.pt", "L_feat.pt", "train.txt", "valid.txt", "test.txt")])
print("input hash:", INPUT_HASH, "| train.py sha:", sha16(f"{W}/train.py"))

release: /kaggle/input/datasets/<kaggle-user>/vimmkg2-release
features: /kaggle/input/datasets/<kaggle-user>/luoi-c-feat-mkg-y/MKG-Y
input hash: f63601aa3c3f8b42 | train.py sha: bed4445a713725d3


## Train (identical flag combination to the reference run; new flag spellings)

In [17]:
EPOCHS = os.environ.get("MENTOR_EPOCHS", "150"); VALID_EVERY = os.environ.get("MENTOR_VALID_EVERY", "5")
SEED = 42
cmd = [sys.executable, "-u", "train.py", "--data_dir", D, "--t_path", f"{D}/T_ctx.pt", "--v_path", f"{D}/V.pt",
       "--use_text", "1", "--use_image", "0", "--exp", f"A3p_MKG-Y_s{SEED}", "--variant", "LSR",
       "--lsr_L", f"{D}/L_feat.pt", "--lsr_gL", "0.15", "--multi_positive", "0",
       "--dump_full_valid", f"{RD}/dump_valid.tsv", "--dump_SL", f"{RD}/SL_2c.pt", "--dump_SL_test", f"{RD}/SL_2c_test.pt",
       "--dump_SL_2c", "1", "--sweep_gL", "0,0.025,0.05,0.1,0.15,0.25,0.4,0.6",
       "--sweep_out", f"{RD}/sweep_gL.tsv", "--sweep_test_out", f"{RD}/sweep_gL_test.tsv",
       "--lsr_off", "RM", "--lsr_l1", "1", "--num_epoch", EPOCHS, "--valid_epoch", VALID_EVERY, "--lr", "5e-4",
       "--batch_size", "512", "--dim_rel", "64", "--reciprocal", "1", "--entity_bias", "0", "--text_mode", "mean",
       "--gate_mode", "gate_off", "--lambda_align", "0.0", "--lambda_gate", "0.0", "--seed", str(SEED)]
json.dump({"cmd": cmd, "input_hash": INPUT_HASH}, open(f"{RD}/config.json", "w"), indent=1)
t0 = time.time()
rc, last = run_streaming(cmd, cwd=W, env=dict(os.environ, VIS_DIM="32"))
print(f"\nreturn code {rc}, {(time.time() - t0) / 60:.1f} min")
assert rc == 0, "training failed; see the output above"

[MENTOR] gate=gate_off seed=42 device=cuda lambda1=0.0 lambda2=0.0
[gate] prior = hard-coded TIER_PRIOR (3 cells)
[features] merged V (15000, 3, 32) -> (15000, 32) (masked mean, 14238/15000 entities have images; mask=V_mask.pt)
[MENTOR] RECIPROCAL: rel 28->56, train 21310->42620
[variant] LSR | paper defaults: {'nguon': 'lexical-semantic router v1', 'dim_L': 64, 'dim_R': 32, 'top_k': 100, 'doi': 's = g_S(q)*S + g_L(q)*L + g_M(q)*M; g initialised ~ (1,0,0); per-query z-score'}
[LSR] active blocks: S + L | g FIXED (1, 0.15, 0), NO R
  use_text=True use_image=False | L_align=False L_gate=False
epoch 1	loss 629.6482	1s
  [gold-invariance] PASS (|delta| tail 0 - head 0, n=256, split=valid)
  [valid e5] MRR 0.3669 H@1 0.2979 H@10 0.4917 *best
  [LSR g e5] S 1.0000 | L 0.1500 | M 0.0000
epoch 10	loss 252.9919	5s
  [gold-invariance] PASS (|delta| tail 0 - head 0, n=256, split=valid)
  [valid e10] MRR 0.4107 H@1 0.3593 H@10 0.4998 *best
  [LSR g e10] S 1.0000 | L 0.1500 | M 0.0000
  [gold-invar

## Score and apply the pre-registered criterion

In [18]:
rc, _ = run_streaming([sys.executable, f"{REL}/scripts/score_a3p_run.py", RD, "--out_json", f"{RD}/score.json"])
res = json.load(open(f"{RD}/score.json"))
ref = None
for l in open(f"{REL}/data/experimental/v3/hits_bang.tsv", encoding="utf-8").read().splitlines()[1:]:
    v = l.split("\t")
    if v[0] == "A3\u2032" and v[1] == "MKG-Y" and v[2] == "t\u1ed5ng" and v[3] == "MRR": ref = float(v[5]) * 100
mrr = 100 * res["test_mrr_at_g_star"]
delta = mrr - ref
print(f"g* = {res['g_star']}   test MRR = {mrr:.2f}   reference = {ref:.2f}   delta = {delta:+.2f}   (informational: original seed-42 run 44.87)")
print("VERDICT:", "PASS" if abs(delta) <= 0.65 else "FAIL", "(criterion |delta| <= 0.65)")
if os.path.isdir("/kaggle/working"):
    subprocess.run(["bash", "-c", f"cd {W} && zip -qrX /kaggle/working/mentor_run_out.zip result -x '*.pt'"], check=False)

{
 "g_star": "0.25",
 "valid_mrr_at_g_star": 0.41984619954605495,
 "test_mrr_at_g_star": 0.44826861089895753,
 "n_test_keys": 5326
}
g* = 0.25   test MRR = 44.83   reference = 44.77   delta = +0.06   (informational: original seed-42 run 44.87)
VERDICT: PASS (criterion |delta| <= 0.65)
